In [1]:
%pip install -q -r ./requirements.txt


Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, json, time, re, hashlib
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import List, Dict, Any, Optional, Tuple

from dotenv import load_dotenv
from pypdf import PdfReader


load_dotenv("../.env")

CONTENT_DIR   = Path(os.getenv("RAG_CONTENT_DIR", "../content"))
CHROMA_DIR    = Path(os.getenv("RAG_CHROMA_DIR", "rag/chroma"))
DOCSETS_META  = Path(os.getenv("RAG_DOCSETS_META", "rag/docsets.json"))
EMB_MODEL_ID  = os.getenv("RAG_EMBEDDING_MODEL", "sentence-transformers/all-MiniLM-L6-v2")

CHROMA_DIR.mkdir(parents=True, exist_ok=True)
DOCSETS_META.parent.mkdir(parents=True, exist_ok=True)
if not DOCSETS_META.exists():
    DOCSETS_META.write_text("{}", encoding="utf-8")

@dataclass
class FileSig:
    name: str
    size: int
    mtime: float

def read_docsets_meta() -> Dict[str, Any]:
    try:
        return json.loads(DOCSETS_META.read_text(encoding="utf-8"))
    except Exception:
        return {}

def write_docsets_meta(data: Dict[str, Any]) -> None:
    DOCSETS_META.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")

def get_lesson_dir(lesson: str) -> Path:
    return (CONTENT_DIR / lesson).resolve()

def collect_pdf_files(lesson: str) -> List[Path]:
    root = get_lesson_dir(lesson)
    if not root.exists():
        return []
    return sorted([p for p in root.rglob("*.pdf") if p.is_file()])

def file_signature(p: Path) -> FileSig:
    st = p.stat()
    return FileSig(name=str(p.name), size=st.st_size, mtime=st.st_mtime)

def compute_docset_hash(files: List[Path]) -> str:
    sigs = [file_signature(p) for p in files]
    sigs_sorted = sorted([asdict(s) for s in sigs], key=lambda d: (d["name"], d["size"], d["mtime"]))
    blob = json.dumps(sigs_sorted, separators=(",", ":"), ensure_ascii=False)
    return hashlib.sha256(blob.encode("utf-8")).hexdigest()


In [3]:
# --- Research-article text filter (heuristic, layout-agnostic) ---
# Keeps main body; trims front matter (title/affiliations/TOC) and tail (references/appendix).
# For robust sectionization later, consider GROBID/Docling. This is a pragmatic starter.

import re
from typing import Optional, List, Dict, Any
from pypdf import PdfReader

# Common "start of body" headings (accept numbered and roman forms)
START_HEADERS = [
    r"^\s*(\d+(\.\d+)*)?\s*Introduction\b",
    r"^\s*[IVX]+\.\s*Introduction\b",
    r"^\s*Background\b",
    r"^\s*(Materials\s+and\s+)?Methods\b",
    r"^\s*Materials\s+and\s+Methods\b",
    r"^\s*Methodology\b",
]

# Common "end of body" headings
END_HEADERS = [
    r"^\s*References\b",
    r"^\s*Bibliography\b",
    r"^\s*Acknowledge?ments?\b",
    r"^\s*Appendix\b",
    r"^\s*Supplementary\b",
]

FRONT_TOKENS = [
    "doi:", "arxiv", "affiliations", "corresponding author", "@",
    "creativecommons", "received", "accepted", "published", "rights reserved"
]

def _read_pdf_pages(path: Path) -> List[str]:
    reader = PdfReader(str(path))
    out = []
    for page in reader.pages:
        try:
            txt = page.extract_text() or ""
        except Exception:
            txt = ""
        out.append(txt.strip())
    return out

def _looks_like_front_matter(text: str) -> bool:
    low = text.lower()
    hits = sum(tok in low for tok in FRONT_TOKENS)
    return hits >= 2 or len(text.split()) < 150  # short & boilerplate-ish

def _find_index(lines: List[str], patterns: List[str]) -> Optional[int]:
    for i, ln in enumerate(lines):
        for pat in patterns:
            if re.search(pat, ln, flags=re.IGNORECASE):
                return i
    return None

def filter_research_pdf(path: Path) -> Dict[str, Any]:
    """
    Returns: {'text': str, 'notes': str, 'skipped_pages': List[int], 'preview': str}
    """
    pages = _read_pdf_pages(path)
    if not pages:
        return {"text": "", "notes": "no_pages", "skipped_pages": [], "preview": ""}

    # Flatten (track page index per line)
    lines, line_page_idx = [], []
    for pi, pg in enumerate(pages):
        for ln in (pg.splitlines() or [""]):
            lines.append(ln.strip())
            line_page_idx.append(pi)

    # Start: look for canonical headings
    start_idx = _find_index(lines, START_HEADERS)

    # Fallback: if not found, skip up to first 3 pages that smell like front matter
    skipped_pages = []
    if start_idx is None:
        for i in range(min(3, len(pages))):
            if _looks_like_front_matter(pages[i]):
                skipped_pages.append(i)
        if skipped_pages:
            first_keep_page = max(skipped_pages) + 1
            for i, pg in enumerate(line_page_idx):
                if pg >= first_keep_page and lines[i]:
                    start_idx = i
                    break

    # End: clip at references/appendix/etc if detected after start
    end_idx = _find_index(lines, END_HEADERS)

    if start_idx is None:
        body = lines
        note = "start=not_found"
    else:
        if end_idx is not None and end_idx > start_idx:
            body = lines[start_idx:end_idx]
            note = "start=ok;end=ok"
        else:
            body = lines[start_idx:]
            note = "start=ok;end=not_found"

    text = "\n".join(body).strip()
    return {
        "text": text,
        "notes": note,
        "skipped_pages": skipped_pages,
        "preview": "\n".join(body[:10])
    }


In [4]:
# --- Splitter, embeddings, vector store wiring ---
import chromadb
from chromadb.utils import embedding_functions
from langchain_text_splitters import RecursiveCharacterTextSplitter

def make_splitter(chunk_size: int = 800, chunk_overlap: int = 100):
    # Paragraph -> line -> sentence-ish -> word-ish fallback
    return RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""],
    )

emb_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name=EMB_MODEL_ID
)

chroma = chromadb.PersistentClient(path=str(CHROMA_DIR))

def collection_for(lesson: str):
    return chroma.get_or_create_collection(
        name=f"{lesson}",
        embedding_function=emb_fn
    )


/home/dimitra/wla/services/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


In [5]:
from fastapi import HTTPException

def ingest_lesson(lesson: str, force: bool = False) -> Dict[str, Any]:
    pdfs = collect_pdf_files(lesson)
    if not pdfs:
        raise HTTPException(status_code=404, detail=f"No PDFs found in {get_lesson_dir(lesson)}")

    h = compute_docset_hash(pdfs)
    meta = read_docsets_meta()
    prev = meta.get(lesson)

    # Skip if unchanged (unless forced)
    if prev and prev.get("hash") == h and not force:
        return {
            "lesson": lesson,
            "docset_hash": h,
            "status": "unchanged",
            "chunks_upserted": 0,
            "files": prev.get("files", []),
        }

    # Fresh collection if hash changed
    col = collection_for(lesson)
    if prev and prev.get("hash") and prev["hash"] != h:
        try:
            chroma.delete_collection(name=col.name)
        except Exception:
            pass
        col = collection_for(lesson)

    splitter = make_splitter()
    docs, metas, ids = [], [], []

    for p in pdfs:
        filt = filter_research_pdf(p)
        body = filt["text"]
        if not body:
            continue

        chunks = splitter.split_text(body)
        for i, ch in enumerate(chunks):
            uid = hashlib.md5(f"{p.name}:{i}:{h}".encode("utf-8")).hexdigest()
            ids.append(f"{p.name}-{i}-{uid}")
            docs.append(ch)
            metas.append({
                "lesson": lesson,
                "source": p.name,
                "chunk_index": i,
                "docset_hash": h,
                "filter_notes": filt["notes"],
            })

    if docs:
        col.upsert(ids=ids, documents=docs, metadatas=metas)

    files_meta = [{"name": p.name, "size": p.stat().st_size, "mtime": p.stat().st_mtime} for p in pdfs]
    meta[lesson] = {
        "hash": h,
        "files": files_meta,
        "collection": col.name,
        "updated_at": int(time.time()),
    }
    write_docsets_meta(meta)

    return {
        "lesson": lesson,
        "docset_hash": h,
        "status": "ingested",
        "chunks_upserted": len(docs),
        "files": files_meta,
    }


In [6]:
from fastapi import FastAPI, HTTPException, Path, Query

# Create the app ONCE
app = FastAPI(title="WLA RAG", version="0.1.0")

# --- routes start here ---

@app.get("/health")
def health():
    return {"ok": True, "service": "fastapi"}

@app.post("/rag/ingest/{lesson}")
def rag_ingest(lesson: str = Path(..., description="Folder under /content"), force: bool = Query(False)):
    return ingest_lesson(lesson, force=force)

@app.get("/rag/docsets/{lesson}")
def rag_docset(lesson: str):
    meta = read_docsets_meta()
    if lesson not in meta:
        raise HTTPException(status_code=404, detail="Unknown lesson")
    return {"lesson": lesson, **meta[lesson]}

@app.get("/rag/stats/{lesson}")
def rag_stats(lesson: str):
    col = collection_for(lesson)
    try:
        n = col.count()
    except Exception:
        n = None
    return {"lesson": lesson, "collection": col.name, "approx_count": n}

In [ ]:
import nest_asyncio, uvicorn, os
from dotenv import load_dotenv
from pathlib import Path

nest_asyncio.apply()
load_dotenv(Path("../.env"))

port = int(os.getenv("RAG_PORT", "8000"))
config = uvicorn.Config(app, host="0.0.0.0", port=port, reload=False, log_level="info")
server = uvicorn.Server(config)
await server.serve()


INFO:     Started server process [533707]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:36558 - "POST /rag/ingest/school_anxiety HTTP/1.1" 200 OK


Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


INFO:     127.0.0.1:33562 - "POST /rag/ingest/school_anxiety HTTP/1.1" 200 OK
